# Integrity and authenticity (HMAC)

# Coding problem, 5.3.c: Implement HMAC-SHA256 by hand

**Skill:** Mitigate Risk · **Maps to:** 5.3.B, 5.3.C · **Language:** Python 3 (pytest)

## Problem
An API needs to verify that requests were sent by a client holding a shared secret key, and
that the request body wasn't tampered with in transit. Rather than calling Python's built-in
`hmac` module, build the FIPS 198-1 HMAC construction yourself on top of `hashlib.sha256`, so
you can see exactly how the ipad/opad key-nesting defeats length-extension attacks.

## Function contract
Implement, in the code editor:
- `hmac_sha256(key: bytes, message: bytes) -> str`, return the HMAC-SHA256 tag as a
  lowercase 64-character hex string.
- `verify(key: bytes, message: bytes, tag: str) -> bool`, recompute the tag and compare it
  (case-insensitively) to the given `tag` using a constant-time comparison.
- `solve(data: dict) -> dict`, the blackbox entry point that dispatches to the two functions
  above (see the Reference section below).

You must implement the ipad/opad key derivation and the double-nested hashing yourself. You
may use `hashlib.sha256` only as the inner hash primitive `H()`, do not import the `hmac`
module for the core logic.

## The construction (FIPS 198-1)
```
HMAC(key, message) = H( (key XOR opad) || H( (key XOR ipad) || message ) )
```
- Block size for SHA-256 is 64 bytes.
- If `key` is longer than 64 bytes, replace it with `H(key)` first.
- Right-pad `key` with `0x00` bytes up to exactly 64 bytes -> call this `k0`.
- `ipad` = the byte `0x36` repeated 64 times; `opad` = the byte `0x5c` repeated 64 times.
- `inner_key = k0 XOR ipad`, `outer_key = k0 XOR opad` (byte-wise XOR).
- `inner_hash = H(inner_key || message)`
- final tag = `H(outer_key || inner_hash)`, hex-encoded.

## Input / Output
- `hmac_sha256(key, message)`: `key: bytes`, `message: bytes` -> hex `str` tag.
- `verify(key, message, tag)`: `tag: str` (hex, any case) -> `bool`.
- `solve(data)`: `data = {"op": "tag", "key": "<hex>", "message": "<str>"}` ->
  `{"tag": "<hex>"}`, or `data = {"op": "verify", "key": "<hex>", "message": "<str>", "tag": "<hex>"}`
  -> `{"valid": <bool>}`.

## Examples
```
hmac_sha256(b"secretkey", b"transfer $100 to acct 42")
  -> "5b1f9a2c...b3" (64 hex chars; must match Python's hmac.new(key, msg, sha256).hexdigest())

verify(b"secretkey", b"transfer $100 to acct 42", "5b1f9a2c...b3") -> True
verify(b"secretkey", b"transfer $900 to acct 42", "5b1f9a2c...b3") -> False  # tampered message
```

## Constraints
- Deterministic, offline, standard library only (`hashlib` only, no `hmac` module).
- Correctness is checked against Python's own `hmac` module inside the tests (used there only
  as an independent oracle, not as part of your implementation).
- Must pass all cases in the tests (visible + hidden), including empty keys/messages, keys
  longer than the 64-byte block size, and case-insensitive tag comparison in `verify()`.

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry points
```python
def hmac_sha256(key: bytes, message: bytes) -> str
def verify(key: bytes, message: bytes, tag: str) -> bool
def solve(data: dict) -> dict
```

`solve()` is the blackbox entry point; `hmac_sha256()` and `verify()` are the underlying
primitives it's built on and are tested directly as well.

### Input
- `hmac_sha256(key, message)`: `key` and `message` are raw `bytes` of any length (including
  empty and longer-than-block-size keys).
- `verify(key, message, tag)`: same `key`/`message` as above; `tag` is a hex string (may be
  upper or lower case) claiming to be the correct HMAC-SHA256 tag.
- `solve(data)`: `data` is a `dict` with `"op"` equal to `"tag"` or `"verify"`.
  - `"tag"`: `{"op": "tag", "key": <hex str>, "message": <str>}`
  - `"verify"`: `{"op": "verify", "key": <hex str>, "message": <str>, "tag": <hex str>}`
  - `key` arrives as a hex string (decode with `bytes.fromhex`); `message` arrives as a
    plain UTF-8 string (encode with `.encode("utf-8")`).

### Output (`result`)
- `hmac_sha256(...)` returns a lowercase hex string (64 hex characters for SHA-256).
- `verify(...)` returns `bool`.
- `solve(...)` returns `{"tag": <hex str>}` for op `"tag"`, or `{"valid": <bool>}` for op `"verify"`.

### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only
  (`hashlib.sha256` may be used as the inner hash primitive `H()`; the ipad/opad key
  derivation and double-nesting must be implemented directly, not via the `hmac` module).
- Must implement the FIPS 198-1 HMAC construction:
  `HMAC(key, message) = H((key XOR opad) || H((key XOR ipad) || message))`, with
  `ipad = 0x36` and `opad = 0x5c` repeated to the 64-byte SHA-256 block size, and keys
  longer than the block size first hashed down with `H()` before padding.
- Any single-bit change to `message` or `key` must produce a different tag.
- `verify()` must accept tags regardless of hex case (upper/lower).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 5.3.c. Student completes the TODOs.
This file intentionally FAILS the tests until implemented."""
import hashlib

BLOCK_SIZE = 64  # SHA-256 block size in bytes
IPAD = 0x36
OPAD = 0x5C


def _h(data: bytes) -> bytes:
    """The inner hash primitive H(). Only stdlib hashlib.sha256 is used here."""
    return hashlib.sha256(data).digest()


def _derive_key(key: bytes) -> bytes:
    """Normalize key to exactly BLOCK_SIZE bytes per FIPS 198-1:
    hash keys longer than the block size, right-pad shorter keys with zero bytes."""
    # TODO: if key is longer than BLOCK_SIZE, replace it with _h(key).
    #       Then right-pad with zero bytes (b"\\x00") up to BLOCK_SIZE.
    raise NotImplementedError("implement _derive_key() — see the problem statement above")


def _xor_bytes(data: bytes, pad: int) -> bytes:
    # TODO: XOR every byte of `data` with the int `pad`; return as bytes.
    raise NotImplementedError("implement _xor_bytes() — see the problem statement above")


def hmac_sha256(key: bytes, message: bytes) -> str:
    """Compute HMAC-SHA256(key, message) and return the tag as a lowercase hex string.

    HMAC(key, message) = H( (key XOR opad) || H( (key XOR ipad) || message ) )
    """
    # TODO: derive k0 with _derive_key(key); build inner_key = k0 XOR ipad and
    #       outer_key = k0 XOR opad; compute inner_hash = H(inner_key || message);
    #       return H(outer_key || inner_hash) as a hex string.
    raise NotImplementedError("implement hmac_sha256() — see the problem statement above")


def _constant_time_equal(a: str, b: str) -> bool:
    """Compare two equal-or-unequal-length strings without short-circuiting on the
    first mismatch, to avoid leaking timing information about where tags differ."""
    # TODO: return False immediately if lengths differ. Otherwise OR together the
    #       XOR of every character pair and return whether the total is zero.
    raise NotImplementedError("implement _constant_time_equal() — see the problem statement above")


def verify(key: bytes, message: bytes, tag: str) -> bool:
    """Return True iff `tag` is the correct HMAC-SHA256 tag for (key, message)."""
    # TODO: recompute the expected tag and compare using _constant_time_equal().
    raise NotImplementedError("implement verify() — see the problem statement above")


def solve(data: dict) -> dict:
    """Blackbox entry point. Dispatches on data['op']:
      - {"op": "tag", "key": <hex str>, "message": <str>}
            -> {"tag": <hex str>}
      - {"op": "verify", "key": <hex str>, "message": <str>, "tag": <hex str>}
            -> {"valid": <bool>}
    Keys and messages are given as UTF-8 strings; `key` is provided as a hex string
    to allow arbitrary binary-looking keys through a JSON-friendly interface.
    """
    # TODO: decode data["key"] with bytes.fromhex, encode data["message"] as utf-8,
    #       then dispatch to hmac_sha256() or verify() depending on data["op"].
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

import hashlib
import hmac as _stdlib_hmac  # only used here, in tests, as an independent oracle


def _oracle_tag(key: bytes, message: bytes) -> str:
    """Independent reference tag computed with Python's stdlib hmac, used only to
    check the student implementation's output, never as part of the implementation."""
    return _stdlib_hmac.new(key, message, hashlib.sha256).hexdigest()


# ---- visible cases -------------------------------------------------------
def test_matches_stdlib_hmac_short_key():
    key = b"secretkey"
    message = b"transfer $100 to acct 42"
    got = impl.hmac_sha256(key, message)
    assert got == _oracle_tag(key, message)


def test_matches_stdlib_hmac_long_key():
    key = b"x" * 100  # longer than the 64-byte block size, exercises key-hashing path
    message = b"hello world"
    got = impl.hmac_sha256(key, message)
    assert got == _oracle_tag(key, message)


def test_tampered_message_changes_tag():
    key = b"secretkey"
    original = b"transfer $100 to acct 42"
    tampered = b"transfer $900 to acct 42"
    tag_original = impl.hmac_sha256(key, original)
    tag_tampered = impl.hmac_sha256(key, tampered)
    assert tag_original != tag_tampered


def test_verify_accepts_correct_tag():
    key = b"apikey123"
    message = b"GET /accounts/42"
    tag = impl.hmac_sha256(key, message)
    assert impl.verify(key, message, tag) is True


def test_verify_rejects_tampered_message():
    key = b"apikey123"
    message = b"GET /accounts/42"
    tag = impl.hmac_sha256(key, message)
    assert impl.verify(key, b"GET /accounts/99", tag) is False


def test_solve_tag_op():
    result = impl.solve({"op": "tag", "key": "deadbeef", "message": "hello"})
    expected = _oracle_tag(bytes.fromhex("deadbeef"), b"hello")
    assert result == {"tag": expected}


def test_solve_verify_op():
    key_hex = "deadbeef"
    tag = _oracle_tag(bytes.fromhex(key_hex), b"hello")
    result = impl.solve({"op": "verify", "key": key_hex, "message": "hello", "tag": tag})
    assert result == {"valid": True}


# ---- hidden / edge cases -------------------------------------------------
def test_wrong_key_fails_verify():
    message = b"payload"
    tag = impl.hmac_sha256(b"key-a", message)
    assert impl.verify(b"key-b", message, tag) is False


def test_empty_message():
    key = b"secretkey"
    got = impl.hmac_sha256(key, b"")
    assert got == _oracle_tag(key, b"")


def test_empty_key():
    message = b"some data"
    got = impl.hmac_sha256(b"", message)
    assert got == _oracle_tag(b"", message)


def test_key_exactly_block_size():
    key = b"k" * 64  # exactly the SHA-256 block size boundary
    message = b"boundary case"
    got = impl.hmac_sha256(key, message)
    assert got == _oracle_tag(key, message)


def test_single_bit_flip_changes_tag_completely():
    key = b"secretkey"
    a = b"password reset for user 1000"
    b = b"password reset for user 1001"  # differs by a single character
    tag_a = impl.hmac_sha256(key, a)
    tag_b = impl.hmac_sha256(key, b)
    assert tag_a != tag_b
    # avalanche: tags should differ in most positions, not just a trailing few
    diff = sum(1 for x, y in zip(tag_a, tag_b) if x != y)
    assert diff > len(tag_a) // 4


def test_verify_case_insensitive_hex_tag():
    key = b"secretkey"
    message = b"case check"
    tag = impl.hmac_sha256(key, message)
    assert impl.verify(key, message, tag.upper()) is True


def test_solve_verify_op_rejects_bad_tag():
    result = impl.solve({"op": "verify", "key": "aa", "message": "x", "tag": "0" * 64})
    assert result == {"valid": False}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
